## Avaliação e melhoria de um sistema RAG

Neste laboratório, iremos avaliar a qualidade de um sistema RAG (Retrieval Augmented Generation) previamente construído utilizando um banco vetorial com PostgreSQL e pgvector. A partir de um conjunto de perguntas de teste, executaremos consultas no sistema e analisaremos as respostas geradas pelo modelo.

Utilizando o framework RAGAS, iremos medir métricas como relevância da resposta, fidelidade ao contexto e qualidade dos documentos recuperados. Em seguida, ajustaremos a estratégia de chunking dos documentos, geraremos novos embeddings e repetiremos a avaliação para comparar os resultados.

Esse processo demonstra como sistemas RAG devem ser avaliados e ajustados continuamente para melhorar a qualidade das respostas.

### Objetivo didático
Demonstrar na prática:
- avaliar um sistema RAG utilizando métricas automáticas
- medir relevância, fidelidade e qualidade do contexto recuperado
- compreender como o retrieval impacta a resposta gerada
- ajustar estratégias de chunking para melhorar o desempenho
- comparar resultados antes e depois das melhorias
- entender o ciclo iterativo de melhoria de sistemas RAG

In [1]:
%%capture
! pip install ragas datasets openai

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

# Credencial embedding e LLM
api_key = os.getenv("OCI_API_KEY")
base_url = "https://inference.generativeai.us-chicago-1.oci.oraclecloud.com/20231130/actions/v1"  # ou sua URL


In [3]:
import warnings

warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    module="pydantic"
)

### Parte 1 - Implementação simples

### Sistema que se quer avaliar

In [4]:
from datasets import Dataset

data = {
 "question": [
  "Why is the sky blue?",
  "What causes rainbows?"
 ],
 "contexts": [
  ["Rayleigh scattering explains blue sky."],
  ["Rainbows form when sunlight refracts and reflects inside raindrops."]
 ],
 "answer": [
  "Blue light scatters more in the atmosphere.",
  "Rainbows occur when light bends and reflects in raindrops."
 ],
 "ground_truth": [
  "Rayleigh scattering causes blue sky.",
  "Rainbows form due to refraction and reflection of light in raindrops."
 ]
}

dataset = Dataset.from_dict(data)

In [5]:
from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI

llm_ragas = LangchainLLMWrapper(
    ChatOpenAI(model="xai.grok-4-fast-non-reasoning", base_url=base_url, api_key=api_key)
)

C:\Users\Amanda Machado\AppData\Local\Temp\ipykernel_30452\772763400.py:4: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  llm_ragas = LangchainLLMWrapper(


In [6]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from IPython.display import display

result = evaluate(
    dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall
    ],
    llm=llm_ragas
)

df = result.to_pandas()


display(df)

Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,Why is the sky blue?,[Rayleigh scattering explains blue sky.],Blue light scatters more in the atmosphere.,Rayleigh scattering causes blue sky.,1.0,0.973167,1.0,1.0
1,What causes rainbows?,[Rainbows form when sunlight refracts and refl...,Rainbows occur when light bends and reflects i...,Rainbows form due to refraction and reflection...,1.0,0.948934,1.0,1.0


### Parte 2 - Avaliando o RAG da aula 2.1 

### Instalação de dependencias

In [7]:
%%capture
!pip install langgraph langchain==1.1.0 langchain-openai==1.1.10 langchain-postgres psycopg[binary] ragas==0.4.1

### Carregando variáveis de ambiente, arquivo `.env`

In [8]:
from dotenv import load_dotenv
import os

load_dotenv()

# Credencial embedding e LLM
api_key = os.getenv("OCI_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")

# Informações do banco
ip_banco = os.getenv("IP_BANCO")
db_password = os.getenv("DB_PASSWORD")
db_name = "vetores"
db_user="appuser"
db_port=5432

### Conectando ao banco vetorial do lab 2.1

In [9]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from openai import OpenAI


client = OpenAI(api_key=openai_api_key)

import psycopg
def get_conn():
    return psycopg.connect(
        host=ip_banco,
        dbname="vetores",
        user="appuser",
        password=db_password,
        port=5432
    )

def busca_banco(pergunta):
    embedding_query = client.embeddings.create(
        model="text-embedding-3-small",
        input=pergunta
    ).data[0].embedding

    conn = get_conn()
    cur = conn.cursor()
    cur.execute("""
    SELECT titulo, conteudo
    FROM documentos
    ORDER BY embedding <=> %s::vector
    LIMIT 5
    """, (embedding_query,))

    resultados = cur.fetchall()
    cur.close()
    conn.close()
    return resultados

In [10]:
llm = ChatOpenAI(model="xai.grok-4-fast-non-reasoning", base_url=base_url, api_key=api_key, n=1)

### Dataset de avaliação

In [15]:
questions = [
    "O que fazer quando o login com 2FA falha?",
    "Quais são as possíveis causas para falha no login com 2FA?",
    "Quanto tempo leva para resolver um problema de 2FA?",
    "O que fazer quando a conta é bloqueada por tentativas inválidas de login?",
    "Qual o prazo para resolver uma conta bloqueada?",
    "O que pode causar erro na emissão de nota fiscal?",
    "Qual o procedimento para resolver erro na emissão de nota fiscal?",
    "O que fazer quando um produto aparece com estoque negativo?",
    "Qual o prazo para corrigir estoque negativo?",
    "Quais passos seguir quando o sistema está lento?",
    "Por que um relatório pode não ser gerado?",
    "O que fazer quando um relatório não gera?",
    "Quais são as causas de erro ao salvar cadastro?",
    "O que fazer quando ocorre erro ao salvar cadastro?",
    "O que pode causar falha em integração externa?",
    "Qual o procedimento para resolver falha em integração externa?",
    "O que fazer quando um e-mail automático não é recebido?",
    "O que fazer quando ocorre erro após atualização de versão?"
]


ground_truth = [
    "Verificar se o horário do celular está automático, gerar um novo código no aplicativo autenticador, evitar reutilizar códigos e, se o erro persistir, solicitar redefinição do 2FA via suporte e configurar novamente o aplicativo.",
    "Código expirado, horário do celular desincronizado ou uso de aplicativo autenticador incorreto.",
    "Até 30 minutos.",
    "Aguardar 15 minutos antes de tentar novamente, abrir chamado interno se permanecer bloqueado, confirmar dados cadastrais, redefinir senha após desbloqueio e confirmar login bem-sucedido.",
    "Até 1 hora.",
    "Certificado digital vencido, token desconectado ou instabilidade no órgão fiscal.",
    "Verificar validade do certificado digital, reconectar o token físico, reiniciar o navegador, testar a emissão novamente e enviar print do erro ao suporte caso continue.",
    "Acessar histórico de movimentações, verificar ordem cronológica dos lançamentos, identificar inconsistências, corrigir o lançamento incorreto e reprocessar o saldo do item.",
    "Até 2 horas.",
    "Testar velocidade da internet, limpar cache e cookies, acessar em aba anônima, testar em outro navegador e registrar o horário do ocorrido.",
    "Filtros incorretos, período sem dados ou timeout de processamento.",
    "Conferir filtros aplicados, reduzir o intervalo de datas, gerar o relatório em formato alternativo, aguardar processamento e registrar chamado com os parâmetros utilizados.",
    "Campo obrigatório não preenchido, formato inválido de dados ou registro duplicado.",
    "Verificar campos obrigatórios, conferir formato de e-mail e CNPJ, confirmar se o registro já existe, corrigir dados e tentar novamente ou enviar print da tela ao suporte.",
    "Token expirado, alteração na API do parceiro ou limite de requisição excedido.",
    "Verificar status da integração no painel, atualizar token de autenticação, testar conexão manual, confirmar dados de acesso e registrar erro completo no chamado.",
    "Verificar pasta de spam, confirmar e-mail cadastrado, solicitar reenvio, adicionar domínio à lista segura e registrar chamado se necessário.",
    "Limpar cache do navegador, fazer logout e login novamente, confirmar a nova versão exibida no rodapé, testar novamente a funcionalidade e reportar inconsistência detalhada."
]

### Executar cenário

In [16]:
answers = []
contexts = []

for question in questions:

    docs = busca_banco(question)
    context_text = [doc[1] for doc in docs]
    context_joined = "\n".join(context_text)

    prompt = f"""
    Responda a pergunta usando apenas as informações do contexto.

    Se a resposta não estiver no contexto, responda:
    "Não encontrei essa informação no documento."

    Contexto:
    {context_joined}

    Pergunta:
    {question}
    """

    response = llm.invoke(prompt)

    answers.append(response.content)
    contexts.append(context_text)

In [17]:

eval_data = {
    "question": questions,
    "answer": answers,
    "contexts": contexts,
    "ground_truth": ground_truth
}

dataset = Dataset.from_dict(eval_data)

result = evaluate(
    dataset=dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall
    ],
    llm=llm_ragas
)

df = result.to_pandas()

display(df)

Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,O que fazer quando o login com 2FA falha?,[MANUAL DE PROTOCOLOS DE ATENDIMENTO\nVERSÃO 2...,### Procedimento para Falha no Login com 2FA\n...,Verificar se o horário do celular está automát...,1.000000,0.879380,0.833333,1.0
1,Quais são as possíveis causas para falha no lo...,[],Não encontrei essa informação no documento.,"Código expirado, horário do celular desincroni...",0.000000,0.000000,0.000000,0.0
2,Quanto tempo leva para resolver um problema de...,[ se o horário do celular está configurado aut...,Prazo estimado: até 30 minutos.,Até 30 minutos.,1.000000,0.834350,1.000000,1.0
3,O que fazer quando a conta é bloqueada por ten...,[],Não encontrei essa informação no documento.,"Aguardar 15 minutos antes de tentar novamente,...",0.000000,0.000000,0.000000,0.0
4,Qual o prazo para resolver uma conta bloqueada?,[],Não encontrei essa informação no documento.,Até 1 hora.,1.000000,0.000000,0.000000,0.0
5,O que pode causar erro na emissão de nota fiscal?,[dido.\n\nPrazo estimado: até 1 hora.\n\n\n===...,Possíveis causas de erro na emissão de nota fi...,"Certificado digital vencido, token desconectad...",1.000000,0.979494,1.000000,1.0
6,Qual o procedimento para resolver erro na emis...,[en físico.\n3. Reiniciar navegador.\n4. Testa...,1. Verificar conexão de internet e firewall.\n...,"Verificar validade do certificado digital, rec...",0.833333,0.818974,1.000000,1.0
7,O que fazer quando um produto aparece com esto...,[],Não encontrei essa informação no documento.,"Acessar histórico de movimentações, verificar ...",0.000000,0.000000,0.000000,0.0
8,Qual o prazo para corrigir estoque negativo?,[dido.\n\nPrazo estimado: até 1 hora.\n\n\n===...,Não encontrei essa informação no documento.,Até 2 horas.,0.000000,0.000000,0.000000,0.0
9,Quais passos seguir quando o sistema está lento?,[mento:\n1. Acessar histórico de movimentações...,1. Verificar histórico de movimentação \n2. C...,"Testar velocidade da internet, limpar cache e ...",1.000000,0.737335,0.000000,0.4


### Parte 3 - Trulens

In [25]:
%%capture
! pip install trulens trulens-providers-openai

In [36]:
import numpy as np
from trulens.core import Metric
from trulens.core import Selector
from trulens.providers.openai import OpenAI

provider = OpenAI(base_url=base_url, api_key=api_key)

f_groundedness = Metric(
    implementation=provider.groundedness_measure_with_cot_reasons_consider_answerability,
    name="Groundedness",
    selectors={
        "source": Selector.select_context(collect_list=True),
        "statement": Selector.select_record_output(),
        "question": Selector.select_record_input(),
    },
)

# Question/answer relevance between overall question and answer.
f_answer_relevance = Metric(
    implementation=provider.relevance_with_cot_reasons,
    name="Answer Relevance",
    selectors={
        "prompt": Selector.select_record_input(),
        "response": Selector.select_record_output(),
    },
)

# Context relevance between question and each context chunk.
f_context_relevance = Metric(
    implementation=provider.context_relevance_with_cot_reasons,
    name="Context Relevance",
    selectors={
        "question": Selector.select_record_input(),
        "context": Selector.select_context(collect_list=False),
    },
    agg=np.mean,  # choose a different aggregation method if you wish
)

In [42]:
from trulens.core import TruSession

session = TruSession()
session.reset_database()

Updating app_name and app_version in apps table: 0it [00:00, ?it/s]
Updating app_id in records table: 0it [00:00, ?it/s]
Updating app_json in apps table: 0it [00:00, ?it/s]


In [38]:
from openai import OpenAI
from trulens.core.otel.instrument import instrument
from trulens.otel.semconv.trace import SpanAttributes

oai_client = OpenAI(api_key=api_key, base_url=base_url)


class RAG:
    def __init__(self, model_name: str = "gpt-5"):
        self.model_name = model_name

    @instrument(
        span_type=SpanAttributes.SpanType.RETRIEVAL,
        attributes={
            SpanAttributes.RETRIEVAL.QUERY_TEXT: "query",
            SpanAttributes.RETRIEVAL.RETRIEVED_CONTEXTS: "return",
        },
    )
    def retrieve(self, query: str) -> list:
        results = busca_banco(query)
        return [doc for sublist in results for doc in sublist]

    @instrument(span_type=SpanAttributes.SpanType.GENERATION)
    def generate_completion(self, query: str, context_list: list) -> str:
        """Generate answer from context with improved prompting."""
        if len(context_list) == 0:
            return "I don't have enough relevant information to answer this question."

        # Join context if it's a list
        context = (
            "\n---\n".join(context_list)
            if isinstance(context_list, list)
            else context_list
        )

        completion = (
            oai_client.chat.completions.create(
                model=self.model_name,
                messages=[
                    {
                        "role": "system",
                        "content": "You are a helpful assistant. ",
                    },
                    {
                        "role": "user",
                        "content": f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer using the context above:",
                    },
                ],
            )
            .choices[0]
            .message.content
        )
        return (
            completion
            if completion
            else "I don't have enough information to answer this question."
        )

    @instrument(
        span_type=SpanAttributes.SpanType.RECORD_ROOT,
        attributes={
            SpanAttributes.RECORD_ROOT.INPUT: "query",
            SpanAttributes.RECORD_ROOT.OUTPUT: "return",
        },
    )
    def query(self, query: str) -> str:
        context_list = self.retrieve(query=query)
        completion = self.generate_completion(
            query=query, context_list=context_list
        )
        return completion

Cannot explicitly set 'record_root' span type, setting to 'unknown'.


In [ ]:
from trulens.apps.app import TruApp

rag = RAG(model_name="xai.grok-4-fast-non-reasoning")

tru_rag = TruApp(
    rag,
    app_name="RAG",
    app_version="base",
    feedbacks=[f_groundedness, f_answer_relevance, f_context_relevance],
)

🦑 Initialized with db url sqlite:///default.sqlite .
🛑 Secret keys may be written to the database. See the `database_redact_keys` option of `TruSession` to prevent this.


Updating app_name and app_version in apps table: 0it [00:00, ?it/s]
Updating app_id in records table: 0it [00:00, ?it/s]
Updating app_json in apps table: 0it [00:00, ?it/s]


✅ experimental Feature.OTEL_TRACING enabled.
🔒 experimental Feature.OTEL_TRACING is enabled and cannot be changed.


C:\Users\Amanda Machado\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\trulens\feedback\llm_provider.py:2717: UserWarning: Failed to process and remove trivial statements. Proceeding with all statements.
  warnings.warn(
C:\Users\Amanda Machado\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
C:\Users\Amanda Machado\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
C:\Users\Amanda Machado\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\trulens\feedba

In [47]:
with tru_rag as recording:
    for query in questions:
        rag.query(query)

In [45]:
session.get_leaderboard()

,,latency,total_cost
app_name,app_version,,
RAG,base,1.582927,0.0
